# Block Matrix Multiplication

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

A live 3D demo: the product cube of A_ik B_kj between A on one wall and B on another; pick a block C_IJ and its column of the cube splits into the two sub-cubes A_I1 B_1J and A_I2 B_2J.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/block-multiplication/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's 4x4 matrices in 2x2 blocks: A's four blocks are scaled by a, b, c, d
a = 1.2
b = 0.6
c = -0.7
d = 1.5
A0 = torch.randn(4, 4, dtype=torch.float64).round(decimals=1)
B = torch.randn(4, 4, dtype=torch.float64).round(decimals=1)
scale = torch.kron(torch.tensor([[a, b], [c, d]], dtype=torch.float64), torch.ones(2, 2, dtype=torch.float64))
A = A0 * scale
C = A @ B
A11, A12, A21, A22 = A[:2, :2], A[:2, 2:], A[2:, :2], A[2:, 2:]
B11, B21 = B[:2, :2], B[2:, :2]
print("C11 = A11 B11 + A12 B21:", torch.allclose(C[:2, :2], A11 @ B11 + A12 @ B21))
print("C11 =\n", C[:2, :2])

# ── The same rule with blocks: each entry becomes a matrix ──────────────────
n = 3                                         # blocks of size 3x3 -> 6x6 matrices
A_big = torch.randn(2 * n, 2 * n, dtype=torch.float64)
B_big = torch.randn(2 * n, 2 * n, dtype=torch.float64)


def blocks(M):
    return M[:n, :n], M[:n, n:], M[n:, :n], M[n:, n:]


A11, A12, A21, A22 = blocks(A_big)
B11, B12, B21, B22 = blocks(B_big)
C_blocks = torch.cat([
    torch.cat([A11 @ B11 + A12 @ B21, A11 @ B12 + A12 @ B22], dim=1),
    torch.cat([A21 @ B11 + A22 @ B21, A21 @ B12 + A22 @ B22], dim=1),
], dim=0)
print("\nblock product == full product:", torch.allclose(C_blocks, A_big @ B_big))

# Order matters inside blocks: A12 @ B21 is not B21 @ A12
print("A12 B21 == B21 A12 ?", torch.allclose(A12 @ B21, B21 @ A12))

# ── Block-diagonal matrices multiply block by block ─────────────────────────
D1, D2 = torch.block_diag(A11, A22), torch.block_diag(B11, B22)
print("block_diag product == block_diag of products:",
      torch.allclose(D1 @ D2, torch.block_diag(A11 @ B11, A22 @ B22)))

C11 = A11 B11 + A12 B21: True
C11 =
 tensor([[-1.6320,  2.5920],
        [-0.1980, -0.4020]], dtype=torch.float64)

block product == full product: True
A12 B21 == B21 A12 ? False
block_diag product == block_diag of products: True
